In [16]:
import pandas as pd
import torch
from torch_geometric.data import HeteroData
from sklearn.model_selection import train_test_split
import numpy as np

# -----------------------
# Load CSVs
# -----------------------

persons = pd.read_csv("person_vertices.csv")          # person_id + features...
conditions = pd.read_csv("condition_vertices.csv")    # condition_concept_id + features...
edges = pd.read_csv("edge_list.csv")                  # columns: person_id, condition_concept_id

# -----------------------
# Create ID maps (PyG requires consecutive IDs)
# -----------------------

person_id_map = {pid: i for i, pid in enumerate(persons['person_id'])}
condition_id_map = {cid: i for i, cid in enumerate(conditions['condition_concept_id'])}

# Map edges to contiguous IDs
edges['pid'] = edges['person_id'].map(person_id_map)
edges['cid'] = edges['condition_concept_id'].map(condition_id_map)

# make boolean columns numeric
for col in persons.columns:
    if persons[col].dtype == bool:
        persons[col] = persons[col].astype(int)

# Extract feature matrices
person_features = torch.tensor(
    persons.drop(columns=['person_id']).values, dtype=torch.float
)

# keep name column as a Python list of labels
condition_labels = conditions['concept_name'].tolist()

# numeric features only
condition_features = torch.tensor(
    conditions.drop(columns=['condition_concept_id', 'concept_name']).values.astype(float),
    dtype=torch.float
)

# -----------------------
# Build HeteroData graph
# -----------------------

data = HeteroData()

data['person'].x = person_features
data['condition'].x = condition_features

edge_index = torch.tensor(edges[['pid', 'cid']].values.T, dtype=torch.long)

data['person', 'has_condition', 'condition'].edge_index = edge_index
